In [1]:
# ================================================================
# CRDL COMPONENT 3 — C3-17 SECTION 1
# PIXEL METROLOGY + CONDITIONAL CALIBRATED 3D MEASUREMENT
# REAL TRAINED MODEL INTEGRATION
#
# NO RE-TRAINING
# NO FILE PICKERS / UPLOADS
# ORIGINAL MODELS AND EXPERIMENTS LEFT UNCHANGED
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import csv, json, hashlib, uuid, shutil, importlib.util

import numpy as np
import torch
from PIL import Image, ImageDraw


# ================================================================
# 1. RESTORE EXACT RESEARCH PATHS
# ================================================================

ROOT = Path("/content/drive/MyDrive")

if not ROOT.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT = (
    ROOT / "crdl_experiments"
    / "component3_fresh_v1"
)

DATA = (
    ROOT / "manushi_research_data"
    / "pothrgbd" / "PData-main"
)

BUNDLE = (
    PROJECT / "08_integration"
    / "C3-14_provisional_component3_"
      "20261021T065928Z_c144de"
)

LABEL_FILE = (
    PROJECT / "04_geometry_targets"
    / "C3-11_section01_RAW_target_pilot_"
      "20261021T043311Z_367db5"
    / "C3-11_section04_weighted_feasibility_"
      "20261021T050356Z_383980"
    / "target_eligibility_manifest.csv"
)

ROLE_FILE = (
    PROJECT / "07_evaluation"
    / "C3-09_physical_evidence_gate_v2_"
      "20261021T024723Z_ec15d662"
    / "C3-10_section02_grouped_pilot_protocol_"
      "20261021T042524Z_735c9f"
    / "PROVISIONAL_pilot_role_manifest.csv"
)

for p in (
    BUNDLE / "selected_model.pt",
    BUNDLE / "component3_service.py",
    BUNDLE / "model_and_inference.py",
    BUNDLE / "bundle_config.json",
    LABEL_FILE,
    ROLE_FILE
):
    assert p.is_file(), f"Missing source: {p}"


# ================================================================
# 2. REPRODUCIBILITY HELPERS
# ================================================================

def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for part in iter(
            lambda: f.read(1048576), b""
        ):
            h.update(part)
    return h.hexdigest()


def readcsv(path):
    with Path(path).open(
        newline="", encoding="utf-8"
    ) as f:
        return list(csv.DictReader(f))


# ================================================================
# 3. CREATE NEW VERSIONED OUTPUT
# ================================================================

RUN = (
    PROJECT / "08_integration"
    / (
        "C3-17_measurement_engine_"
        + datetime.now(
            timezone.utc
        ).strftime("%Y%m%dT%H%M%SZ_")
        + uuid.uuid4().hex[:6]
    )
)

RUN.mkdir(
    parents=True,
    exist_ok=False
)

print("C3-17 measurement run:", RUN)


# ================================================================
# 4. REUSABLE PIXEL AND 3D GEOMETRY ENGINE
# ================================================================

ENGINE_SOURCE = r'''
import numpy as np
from scipy.ndimage import (
    distance_transform_edt,
    binary_erosion,
    label,
    convolve
)
from skimage.morphology import skeletonize


def _mask(x, name):

    a = np.asarray(x)

    if (
        a.ndim != 2
        or (
            not np.issubdtype(a.dtype, np.number)
            and a.dtype != bool
        )
    ):
        raise ValueError(
            f"{name} must be a 2-D numeric/bool array"
        )

    if not np.isfinite(a).all():
        raise ValueError(
            f"{name} has nonfinite values"
        )

    return a > 0


def pixel_measurements(
    pothole_mask,
    crack_mask=None,
    crack_confidence=None
):

    m = _mask(pothole_mask, "pothole_mask")

    if m.size > 12_000_000:
        raise ValueError("Image too large")

    if not m.any():

        return {
            "status": "EMPTY_MASK",
            "footprint_area_px2": 0,
            "components_8": 0,
            "crack": None,
            "physical_units_available": False
        }

    st = np.ones((3, 3), dtype=np.uint8)

    components = int(
        label(m, structure=st)[1]
    )

    boundary = int(
        (
            m & ~binary_erosion(
                m,
                structure=st
            )
        ).sum()
    )

    yy, xx = np.nonzero(m)

    centered = np.stack(
        [
            xx.astype(float) - xx.mean(),
            yy.astype(float) - yy.mean()
        ],
        axis=1
    )

    covariance = (
        centered.T @ centered
        / max(len(centered), 1)
    )

    eigenvectors = np.linalg.eigh(
        covariance
    )[1][:, ::-1]

    projections = centered @ eigenvectors

    spans = np.ptp(
        projections,
        axis=0
    ) + 1.0

    result = {
        "status": "PIXEL_GEOMETRY_ONLY",
        "footprint_area_px2": int(m.sum()),
        "boundary_pixels_approx": boundary,
        "components_8": components,
        "projected_major_extent_px": float(
            max(spans)
        ),
        "projected_minor_extent_px": float(
            min(spans)
        ),
        "physical_units_available": False,
        "crack": None
    }

    if crack_mask is None:

        if crack_confidence is not None:
            raise ValueError(
                "Confidence supplied without crack mask"
            )

        return result

    c = _mask(crack_mask, "crack_mask")

    if c.shape != m.shape:
        raise ValueError(
            "Crack mask shape mismatch"
        )

    avg_conf = None

    if crack_confidence is not None:

        confidence = np.asarray(
            crack_confidence,
            dtype=float
        )

        if (
            confidence.shape != c.shape
            or not np.isfinite(confidence).all()
            or np.any(
                (confidence < 0)
                | (confidence > 1)
            )
        ):
            raise ValueError(
                "Invalid crack confidence map"
            )

        if c.any():

            avg_conf = float(
                np.mean(confidence[c])
            )

    # ------------------------------------------------
    # CRACK SKELETON AND TOPOLOGY
    # ------------------------------------------------

    skeleton = skeletonize(c)

    horizontal = int(
        np.sum(
            skeleton[:, :-1]
            & skeleton[:, 1:]
        )
    )

    vertical = int(
        np.sum(
            skeleton[:-1, :]
            & skeleton[1:, :]
        )
    )

    diagonal1 = int(
        np.sum(
            skeleton[:-1, :-1]
            & skeleton[1:, 1:]
        )
    )

    diagonal2 = int(
        np.sum(
            skeleton[1:, :-1]
            & skeleton[:-1, 1:]
        )
    )

    skeleton_length = float(
        horizontal
        + vertical
        + np.sqrt(2) * (
            diagonal1 + diagonal2
        )
    )

    neighbours = (
        convolve(
            skeleton.astype(np.uint8),
            np.ones((3, 3), dtype=np.uint8),
            mode="constant",
            cval=0
        )
        - skeleton.astype(np.uint8)
    )

    endpoints = int(
        (
            skeleton
            & (neighbours == 1)
        ).sum()
    )

    junction_pixels = (
        skeleton
        & (neighbours >= 3)
    )

    junctions = int(
        label(
            junction_pixels,
            structure=st
        )[1]
    )

    # Approximate crack width from local
    # distance to the segmented crack boundary.

    distance = distance_transform_edt(c)

    widths = (
        2.0 * distance[skeleton]
    )

    result["crack"] = {

        "area_px2": int(c.sum()),

        "components_8": int(
            label(c, structure=st)[1]
        ),

        "skeleton_edge_length_px_approx":
            skeleton_length,

        "skeleton_pixels": int(
            skeleton.sum()
        ),

        "endpoint_pixels_approx":
            endpoints,

        "junction_clusters_approx":
            junctions,

        "median_width_px_approx":
            (
                float(np.median(widths))
                if widths.size else None
            ),

        "p95_width_px_approx":
            (
                float(
                    np.percentile(widths, 95)
                )
                if widths.size else None
            ),

        "mean_supplied_confidence":
            avg_conf,

        "confidence_calibrated": False,

        "length_mm": None,

        "width_mm": None
    }

    return result


# ================================================================
# CAMERA DEPTH -> XYZ
# ================================================================

def xyz_from_depth(
    depth_m,
    fx, fy, cx, cy
):

    z = np.asarray(
        depth_m,
        dtype=np.float64
    )

    if (
        z.ndim != 2
        or not np.isfinite(z).all()
        or (z <= 0).any()
    ):
        raise ValueError(
            "Requires complete positive aligned "
            "metric depth"
        )

    if (
        not all(
            np.isfinite(q)
            for q in (fx, fy, cx, cy)
        )
        or fx <= 0
        or fy <= 0
    ):
        raise ValueError(
            "Invalid camera intrinsics"
        )

    yy, xx = np.indices(z.shape)

    X = (xx - cx) * z / fx

    Y = (yy - cy) * z / fy

    return np.stack(
        [X, Y, z],
        axis=-1
    )


# ================================================================
# CONDITIONAL ROAD-NORMAL METRIC GEOMETRY
# ================================================================

def metric_xyz_measurements(
    xyz_m,
    pothole_mask,
    *,
    evidence
):

    # These evidence flags must come from genuine
    # calibration/provenance records.
    #
    # Passing booleans does not independently
    # certify a real-world dataset.

    required = (
        "xyz_units_m_verified",
        "rgb_xyz_registration_verified",
        "camera_frame_origin_verified",
        "provenance_id"
    )

    if (
        not isinstance(evidence, dict)
        or not all(
            evidence.get(k)
            for k in required
        )
    ):
        raise PermissionError(
            "Meter units, registration, "
            "camera-frame origin and provenance "
            "are required"
        )

    p = np.asarray(
        xyz_m,
        dtype=np.float64
    )

    m = _mask(
        pothole_mask,
        "pothole_mask"
    )

    if (
        p.shape != m.shape + (3,)
        or not m.any()
    ):
        raise ValueError(
            "XYZ must align with nonempty mask"
        )

    valid = (
        np.isfinite(p).all(axis=-1)
        & (p[..., 2] > 0)
    )

    coverage = float(
        (valid & m).sum() / m.sum()
    )

    if coverage < 0.90:
        raise ValueError(
            "Insufficient valid XYZ in pothole"
        )

    # ------------------------------------------------
    # HEALTHY ROAD REFERENCE
    # ------------------------------------------------

    dist = distance_transform_edt(~m)

    ring = (
        (dist >= 8)
        & (dist <= 35)
        & valid
    )

    if ring.sum() < 500:
        raise ValueError(
            "Insufficient healthy-road support"
        )

    support = p[ring]

    if len(support) > 18000:

        indices = np.random.default_rng(
            20261021
        ).choice(
            len(support),
            18000,
            replace=False
        )

        support = support[indices]

    # ------------------------------------------------
    # ROBUST 3D PLANE ESTIMATION
    # ------------------------------------------------

    weights = np.ones(len(support))

    for _ in range(6):

        center = np.average(
            support,
            axis=0,
            weights=weights
        )

        q = support - center

        covariance = (
            (q * weights[:, None]).T @ q
            / max(weights.sum(), 1)
        )

        _, vectors = np.linalg.eigh(
            covariance
        )

        normal = vectors[:, 0]

        # Orient normal toward camera origin.
        if np.dot(normal, -center) < 0:
            normal = -normal

        offset = -float(
            np.dot(normal, center)
        )

        residuals = np.abs(
            support @ normal + offset
        )

        sigma = max(
            1.4826 * np.median(
                np.abs(
                    residuals
                    - np.median(residuals)
                )
            ),
            0.00001
        )

        weights = np.minimum(
            1.0,
            1.345 * sigma
            / (residuals + 1e-12)
        )

    # ------------------------------------------------
    # RAY-PLANE INTERSECTION
    # ------------------------------------------------

    denominator = np.einsum(
        "ijk,k->ij",
        p,
        normal
    )

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):

        factor = -offset / denominator

        reference = (
            p * factor[..., None]
        )

    usable = (
        valid
        & np.isfinite(reference).all(axis=-1)
        & np.isfinite(factor)
        & (factor > 0)
    )

    if (
        (usable & m).sum() / m.sum()
        < 0.90
    ):
        raise ValueError(
            "Invalid ray-plane intersections"
        )

    # ------------------------------------------------
    # ROAD-NORMAL DEPTH
    # ------------------------------------------------

    signed_distance = np.einsum(
        "ijk,k->ij",
        p - reference,
        normal
    )

    depression = np.maximum(
        0.0,
        -signed_distance
    )

    # ------------------------------------------------
    # PROJECTED ROAD-PLANE AREA
    # ------------------------------------------------

    safe = usable.copy()

    safe[0, :] = False
    safe[-1, :] = False
    safe[:, 0] = False
    safe[:, -1] = False

    safe[1:-1, 1:-1] &= (
        usable[:-2, 1:-1]
        & usable[2:, 1:-1]
        & usable[1:-1, :-2]
        & usable[1:-1, 2:]
    )

    Q = np.where(
        usable[..., None],
        reference,
        0.0
    )

    dv, du = np.gradient(
        Q,
        axis=(0, 1)
    )

    area_per_pixel = np.linalg.norm(
        np.cross(du, dv),
        axis=-1
    )

    area_mask = (
        m
        & safe
        & np.isfinite(area_per_pixel)
        & (area_per_pixel > 0)
    )

    if (
        area_mask.sum() / m.sum()
        < 0.90
    ):
        raise ValueError(
            "Insufficient reliable footprint area"
        )

    values = depression[m & usable]

    # Volume approximated by integrating
    # road-normal depression over projected area.

    volume = float(
        np.sum(
            depression[area_mask]
            * area_per_pixel[area_mask]
        )
    )

    return {

        "status":
            "CONDITIONAL_CALIBRATED_INPUT_NOT_FIELD_VALIDATED",

        "source_provenance_id":
            str(evidence["provenance_id"]),

        "pothole_depth_d95_m":
            float(
                np.percentile(values, 95)
            ),

        "pothole_depth_max_m":
            float(values.max()),

        "pothole_depth_mean_m":
            float(values.mean()),

        "projected_plane_area_m2":
            float(
                area_per_pixel[area_mask].sum()
            ),

        "depression_volume_m3":
            volume,

        "mask_xyz_coverage":
            coverage,

        "area_support_fraction":
            float(
                area_mask.sum() / m.sum()
            ),

        "road_fit_healthy_median_abs_m":
            float(
                np.median(
                    np.abs(
                        support @ normal + offset
                    )
                )
            ),

        "physical_accuracy_independently_validated":
            False,

        "road_model":
            "Local 3D plane and "
            "road-normal ray-plane integration"
    }
'''

ENGINE_FILE = (
    RUN / "geometry_measurements.py"
)

ENGINE_FILE.write_text(
    ENGINE_SOURCE,
    encoding="utf-8"
)


# ================================================================
# 5. COPY ORIGINAL TRAINED MODEL INTO NEW BUNDLE
# ================================================================

for filename in (
    "selected_model.pt",
    "model_and_inference.py",
    "component3_service.py",
    "bundle_config.json"
):

    shutil.copy2(
        BUNDLE / filename,
        RUN / filename
    )


# ================================================================
# 6. CREATE THE ENHANCED COMPONENT 3 SERVICE
# ================================================================

SERVICE_SOURCE = r'''
from pathlib import Path
import importlib.util


class Component3MeasurementService:

    def __init__(
        self,
        bundle_root,
        device="cpu"
    ):

        self.root = Path(bundle_root)

        spec = importlib.util.spec_from_file_location(
            "crdl_c317_base",
            self.root / "component3_service.py"
        )

        base_module = importlib.util.module_from_spec(
            spec
        )

        spec.loader.exec_module(
            base_module
        )

        spec2 = importlib.util.spec_from_file_location(
            "crdl_c317_geometry",
            self.root / "geometry_measurements.py"
        )

        geo_module = importlib.util.module_from_spec(
            spec2
        )

        spec2.loader.exec_module(
            geo_module
        )

        self.base = base_module.Component3Prototype(
            self.root,
            device=device
        )

        self.geometry = geo_module

    def predict(
        self,
        rgb,
        pothole_mask,
        crack_mask=None,
        crack_confidence=None,
        include_map=False
    ):

        pixels = self.geometry.pixel_measurements(
            pothole_mask,
            crack_mask,
            crack_confidence
        )

        # Actual trained CRDL inference.
        result = self.base.predict(
            rgb,
            pothole_mask,
            include_map=include_map
        )

        result["measurement_schema"] = (
            "crdl-c3/0.7-pixel-and-conditional-3d"
        )

        result["pixel_measurements"] = pixels

        result["physical_measurement_evidence"] = (
            "NOT_VERIFIED_FOR_POTHRGBD"
        )

        # Physical measurements and severity
        # remain evidence-gated by C3-14.
        return result
'''

SERVICE_FILE = (
    RUN / "component3_measurement_service.py"
)

SERVICE_FILE.write_text(
    SERVICE_SOURCE,
    encoding="utf-8"
)


# ================================================================
# 7. COLD-LOAD THE ENHANCED TRAINED SERVICE
# ================================================================

spec = importlib.util.spec_from_file_location(
    "crdl_c317_service",
    SERVICE_FILE
)

module = importlib.util.module_from_spec(
    spec
)

spec.loader.exec_module(module)

service = module.Component3MeasurementService(
    RUN,
    device=(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
)

geometry = service.geometry

print("Actual trained checkpoint: LOADED")
print("Measurement engine: LOADED")


# ================================================================
# 8. SYNTHETIC CALIBRATED PHYSICAL TEST
# ================================================================

# A known synthetic plane:
# Camera-to-road distance: 1.2 metres
# Maximum depression: 0.035 metres (35 mm)
#
# These intrinsics are synthetic.
# They are NOT PothRGBD camera calibration.

h, w = 192, 256

y, x = np.indices(
    (h, w)
)

mask = (
    ((x - 128) / 42) ** 2
    + ((y - 96) / 31) ** 2
) < 1

crack = (
    (abs(x - 128) < 2)
    & (abs(y - 96) < 29)
)

truth_depth = (
    0.035
    * np.exp(
        -((x - 128) / 30) ** 2
        -((y - 96) / 20) ** 2
    )
    * mask
)

ROAD_DISTANCE = 1.2

FX, FY = 300.0, 305.0

xyz = geometry.xyz_from_depth(
    ROAD_DISTANCE + truth_depth,
    FX,
    FY,
    (w - 1) / 2,
    (h - 1) / 2
)

evidence = {

    "xyz_units_m_verified": True,

    "rgb_xyz_registration_verified": True,

    "camera_frame_origin_verified": True,

    "provenance_id":
        "SYNTHETIC_CALIBRATED_SCENE_ONLY"
}

measured = geometry.metric_xyz_measurements(
    xyz,
    mask,
    evidence=evidence
)

# Known analytic projected pixel area.
pixel_area_expected = (
    (ROAD_DISTANCE / FX)
    * (ROAD_DISTANCE / FY)
)

area_expected = float(
    mask.sum() * pixel_area_expected
)

volume_expected = float(
    np.sum(truth_depth[mask])
    * pixel_area_expected
)

assert abs(
    measured["pothole_depth_max_m"]
    - 0.035
) < 1e-5

assert abs(
    measured["projected_plane_area_m2"]
    - area_expected
) / area_expected < 0.005

assert abs(
    measured["depression_volume_m3"]
    - volume_expected
) / volume_expected < 0.005

# Unsafe calibration must not be accepted.
try:

    geometry.metric_xyz_measurements(
        xyz,
        mask,
        evidence={}
    )

    raise AssertionError(
        "Physical evidence gate failed"
    )

except PermissionError:
    pass

print("Synthetic road-normal depth test: PASS")
print("Synthetic projected area test: PASS")
print("Synthetic volume integration test: PASS")


# ================================================================
# 9. SYNTHETIC TRAINED-SERVICE TEST
# ================================================================

synthetic_rgb = np.zeros(
    (h, w, 3),
    dtype=np.uint8
)

synthetic_rgb[..., 0] = (
    x % 256
).astype(np.uint8)

synthetic_rgb[..., 1] = (
    y % 192
).astype(np.uint8)

synthetic_rgb[..., 2] = 105

output = service.predict(

    synthetic_rgb,

    mask,

    crack_mask=crack,

    crack_confidence=np.full(
        (h, w),
        0.8
    ),

    include_map=False
)

assert (
    output["pixel_measurements"]["crack"]
    ["skeleton_edge_length_px_approx"] > 40
)

assert (
    output["geometry_raw"]["d95_raw"] >= 0
)

assert (
    output["reliability_status"]
    == "REVIEW_ONLY"
)

assert all(

    value is None

    for value in output[
        "physical_measurements"
    ].values()
)


# ================================================================
# 10. EMPTY MASK AND INVALID SHAPE TESTS
# ================================================================

empty = service.predict(

    synthetic_rgb,

    np.zeros(
        (h, w),
        dtype=bool
    )
)

assert (
    empty["reliability_status"]
    == "UNSUPPORTED_EMPTY_MASK"
)

try:

    service.predict(
        synthetic_rgb,
        mask,
        crack_mask=np.ones(
            (4, 4)
        )
    )

    raise AssertionError(
        "Shape safety test failed"
    )

except ValueError:
    pass

print("Pixel crack geometry test: PASS")
print("Empty-mask abstention test: PASS")
print("Invalid-input test: PASS")


# ================================================================
# 11. RECOVER EXISTING STRICT FIT SAMPLES
# ================================================================

roles = {
    r["sample_id"]: r
    for r in readcsv(ROLE_FILE)
}

labels = readcsv(LABEL_FILE)

fit = sorted(

    (
        r for r in labels

        if (
            r["tier"] == "STRICT"

            and roles[
                r["sample_id"]
            ]["temporal_burst_proxy"]
            == "POTHRGBD_TEMPORAL_0004"
        )
    ),

    key=lambda r: r["sample_id"]
)

assert len(fit) == 79


# ================================================================
# 12. REAL SEGMENTATION MASK DECODER
# ================================================================

def yolo_mask(path, width, height):

    canvas = Image.new(
        "L",
        (width, height),
        0
    )

    draw = ImageDraw.Draw(canvas)

    polygons = 0

    for line in path.read_text(
        encoding="utf-8"
    ).splitlines():

        if not line.strip():
            continue

        values = np.fromstring(
            line,
            sep=" ",
            dtype=float
        )

        if not (
            len(values) >= 7
            and (len(values) - 1) % 2 == 0
            and values[0] == 0
            and np.isfinite(values).all()
            and (
                (values[1:] >= 0)
                & (values[1:] <= 1)
            ).all()
        ):
            raise ValueError(
                "Invalid segmentation annotation"
            )

        vertices = [

            (
                round(
                    float(px) * (width - 1)
                ),
                round(
                    float(py) * (height - 1)
                )
            )

            for px, py in values[1:].reshape(
                -1, 2
            )
        ]

        draw.polygon(
            vertices,
            fill=1
        )

        polygons += 1

    if polygons < 1:
        raise ValueError(
            "No class-0 polygon"
        )

    return np.asarray(
        canvas,
        dtype=bool
    )


# ================================================================
# 13. RUN REAL RGB/MASK THROUGH THE FINAL SERVICE
# ================================================================

real_results = []

for row in fit[:3]:

    sid = row["sample_id"]

    role = roles[sid]

    rgb_file = Path(
        role["rgb_path"]
    )

    label_file = Path(
        role["label_path"]
    )

    assert rgb_file.parent.resolve() == (
        DATA / "images"
    ).resolve()

    assert label_file.parent.resolve() == (
        DATA / "labels"
    ).resolve()

    with Image.open(rgb_file) as image:

        rgb = np.asarray(
            image.convert("RGB"),
            dtype=np.uint8
        )

    m = yolo_mask(
        label_file,
        rgb.shape[1],
        rgb.shape[0]
    )

    prediction = service.predict(
        rgb,
        m,
        include_map=False
    )

    assert (
        prediction["reliability_status"]
        == "REVIEW_ONLY"
    )

    assert (
        prediction["pixel_measurements"]
        ["footprint_area_px2"]
        == int(m.sum())
    )

    assert all(

        value is None

        for value in prediction[
            "physical_measurements"
        ].values()
    )

    real_results.append({

        "sample_id": sid,

        "footprint_px2":
            prediction["pixel_measurements"]
            ["footprint_area_px2"],

        "predicted_axial_d95_raw":
            prediction["geometry_raw"]
            ["d95_raw"],

        "status":
            prediction["reliability_status"],

        "physical_units_verified":
            False
    })

print(
    "Real training-group model tests:",
    len(real_results),
    "PASS"
)


# ================================================================
# 14. SAVE RESEARCH EVIDENCE
# ================================================================

report = {

    "stage":
        "C3-17_SECTION01",

    "model_source_bundle":
        str(BUNDLE),

    "real_fit_inference_count":
        len(real_results),

    "synthetic_physical_geometry_test":
        "PASS",

    "synthetic_truth_peak_depth_m":
        0.035,

    "synthetic_recovered_peak_m":
        measured["pothole_depth_max_m"],

    "synthetic_expected_area_m2":
        area_expected,

    "synthetic_recovered_area_m2":
        measured["projected_plane_area_m2"],

    "synthetic_expected_volume_m3":
        volume_expected,

    "synthetic_recovered_volume_m3":
        measured["depression_volume_m3"],

    "real_sensor_scale_certified":
        False,

    "real_intrinsics_certified":
        False,

    "real_3d_volume_available":
        False,

    "physical_severity_enabled":
        False,

    "C1_C2_teammate_prediction_handoff_verified":
        False,

    "real_3d_policy":
        "No metric area or volume from "
        "uncalibrated PothRGBD or Fan data",

    "source_checkpoint_sha256":
        sha(BUNDLE / "selected_model.pt")
}

(RUN / "measurement_evidence.json").write_text(

    json.dumps(
        report,
        indent=2
    ),

    encoding="utf-8"
)

(RUN / "synthetic_metric_measurements.json").write_text(

    json.dumps(
        measured,
        indent=2
    ),

    encoding="utf-8"
)

(RUN / "real_training_input_smoke.json").write_text(

    json.dumps(
        real_results,
        indent=2
    ),

    encoding="utf-8"
)


# ================================================================
# 15. SAVE COMPONENT 3 API CONTRACT
# ================================================================

api = {

    "api":
        "Component3MeasurementService(root, device)"
        ".predict(rgb, pothole_mask, crack_mask=None, "
        "crack_confidence=None, include_map=False)",

    "input_rgb":
        "HxWx3 uint8 RGB",

    "mask":
        "HxW aligned binary pothole mask",

    "crack_confidence":
        "Optional HxW array within 0..1; "
        "not calibrated",

    "output":
        "Trained raw axial depression, "
        "pixel-domain pothole/crack measurements, "
        "reliability and abstention",

    "pixel_units":
        "px and px^2; not metric",

    "physical_depth_area_volume":
        "Null on real current inputs",

    "calibrated_xyz_helper":
        "metric_xyz_measurements(xyz_m, mask, evidence=...)"
        " for documented calibrated XYZ only",

    "physical_accuracy_claim":
        "NOT ESTABLISHED"
}

(RUN / "api_contract.json").write_text(

    json.dumps(
        api,
        indent=2
    ),

    encoding="utf-8"
)


# ================================================================
# 16. CHECKSUM MANIFEST
# ================================================================

receipts = []

for path in sorted(
    RUN.iterdir()
):

    if (
        path.is_file()
        and path.name != "sha256_manifest.json"
    ):

        receipts.append({

            "file": path.name,

            "bytes": path.stat().st_size,

            "sha256": sha(path)
        })

(RUN / "sha256_manifest.json").write_text(

    json.dumps(
        receipts,
        indent=2
    ),

    encoding="utf-8"
)

assert (
    sha(BUNDLE / "selected_model.pt")
    == report["source_checkpoint_sha256"]
)


# ================================================================
# 17. FINAL OUTPUT
# ================================================================

print("\n" + "=" * 70)

print(
    "C3-17 SECTION 1 — MEASUREMENT ENGINE COMPLETE"
)

print("=" * 70)

print(
    "Synthetic calibrated XYZ geometry: PASS"
)

print(
    "Truth vs recovered maximum depth (m):",
    0.035,
    measured["pothole_depth_max_m"]
)

print(
    "Truth vs recovered area (m2):",
    area_expected,
    measured["projected_plane_area_m2"]
)

print(
    "Truth vs recovered volume (m3):",
    volume_expected,
    measured["depression_volume_m3"]
)

print(
    "Real training-group RGB/mask service tests:",
    len(real_results)
)

print(
    "Pixel geometry and crack measurements: PASS"
)

print(
    "Invalid-input and empty-mask tests: PASS"
)

print(
    "PothRGBD capture calibration proven: NO"
)

print(
    "Physical severity enabled: NO"
)

print(
    "Teammate model handoff completed: NO"
)

print("\nSaved:")
print(RUN)

print("=" * 70)

Mounted at /content/drive
C3-17 measurement run: /content/drive/MyDrive/crdl_experiments/component3_fresh_v1/08_integration/C3-17_measurement_engine_20261021T084756Z_a2b0e5
Actual trained checkpoint: LOADED
Measurement engine: LOADED
Synthetic road-normal depth test: PASS
Synthetic projected area test: PASS
Synthetic volume integration test: PASS
Pixel crack geometry test: PASS
Empty-mask abstention test: PASS
Invalid-input test: PASS
Real training-group model tests: 3 PASS

C3-17 SECTION 1 — MEASUREMENT ENGINE COMPLETE
Synthetic calibrated XYZ geometry: PASS
Truth vs recovered maximum depth (m): 0.035 0.034999999999859366
Truth vs recovered area (m2): 0.06413114754098362 0.06413114754099863
Truth vs recovered volume (m3): 0.0009173559710858227 0.0009173559710770234
Real training-group RGB/mask service tests: 3
Pixel geometry and crack measurements: PASS
Invalid-input and empty-mask tests: PASS
PothRGBD capture calibration proven: NO
Physical severity enabled: NO
Teammate model handoff

In [2]:
# ================================================================
# CRDL COMPONENT 3 — C3-18 SECTION 1
# RELIABILITY/SEVERITY GATES + C1/C2 HANDOFF PROTOTYPE
#
# ONE COLAB CELL — NO UPLOADS — NO NEW TRAINING
# EXISTING EXPERIMENTS LEFT UNCHANGED
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import csv, json, hashlib, uuid, shutil, importlib.util

import numpy as np
import torch
from PIL import Image, ImageDraw


# ================================================================
# 1. RESTORE EXISTING RESEARCH PATHS
# ================================================================

ROOT = Path('/content/drive/MyDrive')

if not ROOT.is_dir():
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = (
    ROOT / 'crdl_experiments'
    / 'component3_fresh_v1'
)

DATA = (
    ROOT / 'manushi_research_data'
    / 'pothrgbd' / 'PData-main'
)

SOURCE = (
    PROJECT / '08_integration'
    / 'C3-17_measurement_engine_20261021T084756Z_a2b0e5'
)

ROLE_FILE = (
    PROJECT / '07_evaluation'
    / 'C3-09_physical_evidence_gate_v2_20261021T024723Z_ec15d662'
    / 'C3-10_section02_grouped_pilot_protocol_20261021T042524Z_735c9f'
    / 'PROVISIONAL_pilot_role_manifest.csv'
)

LABEL_FILE = (
    PROJECT / '04_geometry_targets'
    / 'C3-11_section01_RAW_target_pilot_20261021T043311Z_367db5'
    / 'C3-11_section04_weighted_feasibility_20261021T050356Z_383980'
    / 'target_eligibility_manifest.csv'
)

FILES = [
    'selected_model.pt',
    'model_and_inference.py',
    'component3_service.py',
    'component3_measurement_service.py',
    'geometry_measurements.py',
    'bundle_config.json'
]

assert all(
    (SOURCE / n).is_file()
    for n in FILES
)

assert ROLE_FILE.is_file()
assert LABEL_FILE.is_file()


# ================================================================
# 2. REPRODUCIBILITY HELPERS
# ================================================================

def sha(p):

    h = hashlib.sha256()

    with Path(p).open('rb') as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b''
        ):
            h.update(chunk)

    return h.hexdigest()


def csvrows(p):

    with Path(p).open(
        newline='',
        encoding='utf8'
    ) as f:

        return list(csv.DictReader(f))


def savejson(p, data):

    Path(p).write_text(
        json.dumps(
            data,
            indent=2,
            allow_nan=False
        ),
        encoding='utf8'
    )


# ================================================================
# 3. CREATE NEW VERSIONED RELEASE-CANDIDATE DIRECTORY
# ================================================================

RUN = (
    PROJECT / '08_integration'
    / (
        'C3-18_reliability_C1C2_handoff_'
        + datetime.now(
            timezone.utc
        ).strftime('%Y%m%dT%H%M%SZ_')
        + uuid.uuid4().hex[:6]
    )
)

RUN.mkdir(
    parents=True,
    exist_ok=False
)

original_hashes = {
    name: sha(SOURCE / name)
    for name in FILES
}

for name in FILES:

    shutil.copy2(
        SOURCE / name,
        RUN / name
    )

    assert sha(RUN / name) == original_hashes[name]


# ================================================================
# 4. IMPLEMENT THE REUSABLE INTEGRATION SERVICE
# ================================================================

POLICY_SOURCE = r'''
from pathlib import Path
import importlib.util
import numpy as np

POLICY_VERSION = 'C3-18.1.0'
SCHEMA_VERSION = 'crdl-c3/0.8-development'


def _binary(a, shape, field):

    a = np.asarray(a)

    if a.ndim != 2 or a.shape != tuple(shape):

        raise ValueError(
            f'{field} must be a 2D mask of shape {shape}'
        )

    if (
        a.dtype != bool
        and not np.issubdtype(a.dtype, np.number)
    ):

        raise ValueError(
            f'{field} must be numeric or bool'
        )

    if not np.isfinite(a).all():

        raise ValueError(
            f'{field} contains non-finite values'
        )

    # No silent thresholding of probability maps.
    if not np.all(
        (a == 0) | (a == 1) | (a == 255)
    ):

        raise ValueError(
            f'{field} requires binary 0/1 or 0/255; '
            'threshold C2 scores explicitly'
        )

    return a != 0


def _confidence(v, field):

    if v is None:
        return None

    if isinstance(v, (bool, np.bool_)):

        raise ValueError(
            f'{field} must be numeric'
        )

    if not np.isscalar(v):

        raise ValueError(
            f'{field} must be a scalar in [0,1]'
        )

    value = float(v)

    if not np.isfinite(value) or not 0 <= value <= 1:

        raise ValueError(
            f'{field} must be in [0,1]'
        )

    return value


def _rgb(a):

    a = np.asarray(a)

    if (
        a.ndim != 3
        or a.shape[2] != 3
        or a.dtype != np.uint8
        or a.shape[0] < 8
        or a.shape[1] < 8
        or a.shape[0] * a.shape[1] > 12_000_000
    ):

        raise ValueError(
            'RGB must be HxWx3 uint8 '
            'within supported size'
        )

    return a


def _bbox(box, w, h):

    if not isinstance(box, (list, tuple)) or len(box) != 4:

        raise ValueError(
            'bbox_xyxy requires four pixel integers'
        )

    if any(
        isinstance(v, (bool, np.bool_))
        or not isinstance(v, (int, np.integer))
        for v in box
    ):

        raise ValueError(
            'bbox_xyxy must contain integers'
        )

    x0, y0, x1, y1 = map(int, box)

    if not (
        0 <= x0 < x1 <= w
        and 0 <= y0 < y1 <= h
    ):

        raise ValueError(
            'bbox is outside image or empty'
        )

    return x0, y0, x1, y1


class Component3Handoff:

    def __init__(self, bundle_path, device='cpu'):

        self.root = Path(bundle_path)

        spec = importlib.util.spec_from_file_location(
            'crdl_c318_measurement',
            self.root / 'component3_measurement_service.py'
        )

        mod = importlib.util.module_from_spec(spec)

        spec.loader.exec_module(mod)

        self.backend = mod.Component3MeasurementService(
            self.root,
            device=device
        )

    def predict_frame(
        self,
        rgb,
        defects,
        *,
        include_maps=False
    ):

        frame = _rgb(rgb)

        h, w = frame.shape[:2]

        if (
            not isinstance(defects, (list, tuple))
            or len(defects) > 100
        ):

            raise ValueError(
                'defects must be a list with <=100 entries'
            )

        used_ids = set()
        outputs = []

        for defect in defects:

            if not isinstance(defect, dict):

                raise ValueError(
                    'Each defect must be an object'
                )

            defect_id = defect.get('defect_id')

            if (
                not isinstance(defect_id, str)
                or not defect_id
                or defect_id in used_ids
                or len(defect_id) > 128
            ):

                raise ValueError(
                    'defect_id must be unique and nonempty'
                )

            used_ids.add(defect_id)

            space = defect.get('mask_space')

            if space not in ('frame', 'bbox'):

                raise ValueError(
                    'mask_space must be frame or bbox'
                )

            bbox = _bbox(
                defect.get('bbox_xyxy'),
                w,
                h
            )

            x0, y0, x1, y1 = bbox

            # ----------------------------------------
            # Reconstruct full-frame mask
            # ----------------------------------------

            if space == 'bbox':

                roi = _binary(
                    defect['mask'],
                    (y1-y0, x1-x0),
                    'ROI pothole mask'
                )

                full_mask = np.zeros(
                    (h, w),
                    dtype=bool
                )

                full_mask[y0:y1, x0:x1] = roi

            else:

                full_mask = _binary(
                    defect['mask'],
                    (h, w),
                    'frame pothole mask'
                )

            # ----------------------------------------
            # Optional associated crack mask
            # ----------------------------------------

            crack = defect.get('crack_mask')

            confidence_map = defect.get(
                'crack_confidence_map'
            )

            if crack is not None:

                crack = _binary(
                    crack,
                    (h, w),
                    'associated crack mask'
                )

            if confidence_map is not None:

                confidence_map = np.asarray(
                    confidence_map,
                    dtype=float
                )

                if (
                    crack is None
                    or confidence_map.shape != (h, w)
                    or not np.isfinite(
                        confidence_map
                    ).all()
                    or (confidence_map < 0).any()
                    or (confidence_map > 1).any()
                ):

                    raise ValueError(
                        'Crack confidence map requires '
                        'aligned crack mask and values [0,1]'
                    )

            detection_confidence = _confidence(
                defect.get('detection_confidence'),
                'detection_confidence'
            )

            segmentation_confidence = _confidence(
                defect.get('segmentation_confidence'),
                'segmentation_confidence'
            )

            # ----------------------------------------
            # ACTUAL TRAINED CRDL INFERENCE
            # ----------------------------------------

            result = self.backend.predict(
                frame,
                full_mask,
                crack_mask=crack,
                crack_confidence=confidence_map,
                include_map=include_maps
            )

            # ----------------------------------------
            # INPUT QUALITY DIAGNOSTICS
            # ----------------------------------------

            mask_count = int(full_mask.sum())

            flags = []

            if mask_count == 0:

                flags.append(
                    'EMPTY_DEFECT_MASK'
                )

            else:

                if mask_count < 200:

                    flags.append(
                        'SMALL_MASK_HEURISTIC'
                    )

                if mask_count / (h * w) > 0.60:

                    flags.append(
                        'LARGE_MASK_HEURISTIC'
                    )

                if (
                    full_mask[0].any()
                    or full_mask[-1].any()
                    or full_mask[:, 0].any()
                    or full_mask[:, -1].any()
                ):

                    flags.append(
                        'MASK_TOUCHES_FRAME_BORDER'
                    )

            if detection_confidence is None:

                flags.append(
                    'DETECTION_CONFIDENCE_NOT_PROVIDED'
                )

            if segmentation_confidence is None:

                flags.append(
                    'SEGMENTATION_CONFIDENCE_NOT_PROVIDED'
                )

            if full_mask.any() and space == 'frame':

                exterior = full_mask.copy()

                exterior[y0:y1, x0:x1] = False

                if exterior.any():

                    flags.append(
                        'MASK_EXTENDS_OUTSIDE_SUPPLIED_BBOX'
                    )

            # ----------------------------------------
            # RELIABILITY DECISION
            # ----------------------------------------

            if mask_count == 0:

                reliability = 'UNSUPPORTED_EMPTY_MASK'

                action = 'CORRECT_INPUT'

            elif any(
                f in flags
                for f in (
                    'SMALL_MASK_HEURISTIC',
                    'LARGE_MASK_HEURISTIC',
                    'MASK_TOUCHES_FRAME_BORDER',
                    'MASK_EXTENDS_OUTSIDE_SUPPLIED_BBOX'
                )
            ):

                reliability = 'REVIEW_REQUIRED_INPUT_QA'

                action = 'REVIEW'

            else:

                reliability = 'REVIEW_ONLY_UNCALIBRATED'

                action = 'REVIEW'

            # These are input QA heuristics.
            # They are NOT calibrated risk rules.

            result['schema_version'] = SCHEMA_VERSION

            result['defect_id'] = defect_id

            result['source_bbox_xyxy'] = list(bbox)

            result['mask_space_received'] = space

            result['reliability_status'] = reliability

            result['reliability'] = {

                'policy_version': POLICY_VERSION,

                'action': action,

                'input_qa_flags': flags,

                'confidence_inputs_are_uncalibrated':
                    True,

                'detection_confidence_reported':
                    detection_confidence,

                'segmentation_confidence_reported':
                    segmentation_confidence,

                'independent_scene_validation':
                    False,

                'physical_scale_verified':
                    False,

                'metric_measurement_eligible':
                    False
            }

            # ----------------------------------------
            # SAFE SEVERITY ABSTENTION
            # ----------------------------------------

            result['severity'] = {

                'physical_level': None,

                'visual_level': None,

                'status':
                    'ABSTAIN_NO_VALIDATED_SEVERITY_THRESHOLDS',

                'risk_score': None
            }

            result['physical_measurements'] = {

                'depth_d95_mm': None,

                'depth_max_mm': None,

                'area_m2': None,

                'volume_m3': None,

                'crack_length_mm': None,

                'crack_width_mm': None
            }

            result['measurement_notes'] = (
                'Geometry RAW = unverified '
                'PothRGBD-style axial residual; '
                'pixel geometry is not physical area/length; '
                'no validated severity.'
            )

            outputs.append(result)

        return {

            'schema_version': SCHEMA_VERSION,

            'frame_height': h,

            'frame_width': w,

            'defect_count': len(outputs),

            'defects': outputs,

            'overall_status':
                'REVIEW_ONLY_NOT_FIELD_VALIDATED',

            'physical_calibration_verified':
                False,

            'teammate_model_handoff_verified':
                False
        }
'''

POLICY_FILE = (
    RUN / 'component3_handoff_service.py'
)

POLICY_FILE.write_text(
    POLICY_SOURCE,
    encoding='utf8'
)


# ================================================================
# 5. COLD-LOAD THE COMPLETE ENHANCED SERVICE
# ================================================================

spec = importlib.util.spec_from_file_location(
    'crdl_c318_handoff',
    POLICY_FILE
)

mod = importlib.util.module_from_spec(spec)

spec.loader.exec_module(mod)

service = mod.Component3Handoff(
    RUN,
    device=(
        'cuda'
        if torch.cuda.is_available()
        else 'cpu'
    )
)

print('Actual C3-14 trained checkpoint: COLD-LOADED')


# ================================================================
# 6. SYNTHETIC SOFTWARE-CONTRACT TESTS
# ================================================================

h, w = 192, 256

Y, X = np.indices((h, w))

rgb = np.zeros(
    (h, w, 3),
    dtype=np.uint8
)

rgb[..., 0] = X.astype(np.uint8)
rgb[..., 1] = Y.astype(np.uint8)
rgb[..., 2] = 85

box = [50, 35, 190, 145]

roi = (
    ((X[35:145, 50:190] - 120) / 38) ** 2
    + ((Y[35:145, 50:190] - 90) / 30) ** 2
) < 1

full = np.zeros(
    (h, w),
    dtype=bool
)

full[35:145, 50:190] = roi

crack = np.zeros(
    (h, w),
    dtype=np.uint8
)

crack[85:110, 119:122] = 1


# ================================================================
# 7. TEST FULL-FRAME C1/C2 INPUT
# ================================================================

metadata = {

    'defect_id': 'SYNTHETIC_1',

    'bbox_xyxy': box,

    'mask_space': 'frame',

    'mask': full,

    'crack_mask': crack,

    'crack_confidence_map':
        np.full((h, w), 0.75),

    'detection_confidence': 0.88,

    'segmentation_confidence': 0.82
}

frame_out = service.predict_frame(
    rgb,
    [metadata]
)

assert frame_out['defect_count'] == 1

one = frame_out['defects'][0]

assert one['geometry_raw']['d95_raw'] >= 0

assert (
    one['pixel_measurements']['footprint_area_px2']
    == int(full.sum())
)

assert (
    one['pixel_measurements']['crack']
    ['skeleton_edge_length_px_approx'] > 0
)

assert (
    one['physical_measurements']['depth_d95_mm']
    is None
)

assert one['severity']['physical_level'] is None

assert one['reliability_status'] == (
    'REVIEW_ONLY_UNCALIBRATED'
)


# ================================================================
# 8. TEST BOUNDING-BOX MASK REPROJECTION
# ================================================================

roi_metadata = dict(
    metadata,
    mask_space='bbox',
    mask=roi
)

roi_out = service.predict_frame(
    rgb,
    [roi_metadata]
)['defects'][0]

assert (
    roi_out['pixel_measurements']['footprint_area_px2']
    == int(full.sum())
)

assert np.isclose(

    roi_out['geometry_raw']['d95_raw'],

    one['geometry_raw']['d95_raw'],

    rtol=0,

    atol=1e-6
)


# ================================================================
# 9. TEST EMPTY DEFECT HANDLING
# ================================================================

empty = dict(

    metadata,

    defect_id='EMPTY',

    mask=np.zeros(
        (h, w),
        dtype=bool
    )
)

e = service.predict_frame(
    rgb,
    [empty]
)['defects'][0]

assert e['reliability_status'] == (
    'UNSUPPORTED_EMPTY_MASK'
)

assert e['geometry_raw'] is None


# ================================================================
# 10. TEST INVALID INPUT REJECTION
# ================================================================

invalid_tests = [

    dict(
        metadata,
        defect_id='BAD1',
        mask=np.full(
            (h, w),
            0.35
        )
    ),

    dict(
        metadata,
        defect_id='BAD2',
        detection_confidence=1.4
    ),

    dict(
        metadata,
        defect_id='BAD3',
        mask_space='bbox',
        mask=np.zeros(
            (3, 4)
        )
    )
]

for candidate in invalid_tests:

    try:

        service.predict_frame(
            rgb,
            [candidate]
        )

    except ValueError:

        pass

    else:

        raise AssertionError(
            'Invalid input was accepted: '
            + candidate['defect_id']
        )

print(
    'Frame/ROI equivalence, invalid inputs, '
    'empty mask, crack inputs and abstention: PASS'
)


# ================================================================
# 11. RECOVER REAL TRAINING-ROLE EXAMPLES
# ================================================================

roles = {

    r['sample_id']: r

    for r in csvrows(ROLE_FILE)
}

labels = csvrows(LABEL_FILE)

fit = sorted(

    [

        r for r in labels

        if (
            r['tier'] == 'STRICT'

            and roles[
                r['sample_id']
            ]['temporal_burst_proxy']
            == 'POTHRGBD_TEMPORAL_0004'
        )
    ],

    key=lambda r: r['sample_id']
)

assert len(fit) == 79


# ================================================================
# 12. DECODE REAL YOLO SEGMENTATION ANNOTATIONS
# ================================================================

def yolo_mask(path, shape):

    h, w = shape

    im = Image.new(
        'L',
        (w, h),
        0
    )

    draw = ImageDraw.Draw(im)

    count = 0

    for line in Path(path).read_text(
        encoding='utf8'
    ).splitlines():

        parts = line.strip().split()

        if not parts:
            continue

        nums = np.array(
            [float(p) for p in parts],
            dtype=float
        )

        if not (
            len(nums) >= 7
            and (len(nums) - 1) % 2 == 0
            and np.isfinite(nums).all()
            and nums[0] == 0
            and np.all(
                (nums[1:] >= 0)
                & (nums[1:] <= 1)
            )
        ):

            raise ValueError(
                'Invalid YOLO polygon'
            )

        coords = nums[1:].reshape(-1, 2)

        draw.polygon(

            [

                (
                    round(a * (w - 1)),
                    round(b * (h - 1))
                )

                for a, b in coords
            ],

            fill=1
        )

        count += 1

    if count < 1:

        raise ValueError(
            'No class-0 polygon'
        )

    return np.asarray(
        im,
        dtype=bool
    )


# ================================================================
# 13. THREE REAL RGB + MASK HANDOFF TESTS
# ================================================================

real = []

for r in fit[:3]:

    sid = r['sample_id']

    src = roles[sid]

    im_path = Path(
        src['rgb_path']
    )

    lab_path = Path(
        src['label_path']
    )

    assert im_path.parent.resolve() == (
        DATA / 'images'
    ).resolve()

    assert lab_path.parent.resolve() == (
        DATA / 'labels'
    ).resolve()

    with Image.open(im_path) as img:

        image = np.asarray(
            img.convert('RGB'),
            dtype=np.uint8
        )

    mask = yolo_mask(
        lab_path,
        image.shape[:2]
    )

    ys, xs = np.where(mask)

    bbox = [

        int(xs.min()),

        int(ys.min()),

        int(xs.max()) + 1,

        int(ys.max()) + 1
    ]

    result = service.predict_frame(

        image,

        [
            {
                'defect_id': sid,

                'bbox_xyxy': bbox,

                'mask_space': 'frame',

                'mask': mask
            }
        ]

    )['defects'][0]

    assert (
        result['geometry_raw']['d95_raw'] >= 0
    )

    assert (
        result['pixel_measurements']
        ['footprint_area_px2']
        == int(mask.sum())
    )

    assert (
        result['physical_measurements']['area_m2']
        is None
    )

    assert (
        result['severity']['physical_level']
        is None
    )

    real.append({

        'sample_id': sid,

        'mask_area_px2': int(mask.sum()),

        'predicted_d95_raw': float(
            result['geometry_raw']['d95_raw']
        ),

        'reliability_status':
            result['reliability_status']
    })


# ================================================================
# 14. SAVE THE FROZEN RELIABILITY POLICY
# ================================================================

config = {

    'stage':
        'C3-18_SECTION01_HANDOFF_AND_RELIABILITY',

    'source_c317_bundle':
        str(SOURCE),

    'source_sha256':
        original_hashes,

    'trained_geometry_selected':
        'DIRECT_RGB_MASK',

    'physical_depth_area_volume':
        None,

    'real_physical_severity':
        None,

    'visual_severity':
        None,

    'confidence_calibrated':
        False,

    'independent_scene_generalization_verified':
        False,

    'C1_C2_actual_teammate_outputs_tested':
        False,

    'supported_mask_spaces':
        ['frame', 'bbox'],

    'segmentation_thresholding':
        'Caller must explicitly binarize mask',

    'bbox_convention':
        'Integer xyxy, half-open, RGB coordinates',

    'input_qa_heuristics_not_validated_risk_rules': {

        'min_mask_pixels': 200,

        'max_mask_fraction': 0.60,

        'border_touch': True
    },

    'status_semantics': {

        'UNSUPPORTED_EMPTY_MASK':
            'No valid defect segmentation',

        'REVIEW_REQUIRED_INPUT_QA':
            'Mask QA heuristic triggered',

        'REVIEW_ONLY_UNCALIBRATED':
            'RAW/pixel outputs available; '
            'physical measurements unsupported'
    },

    'scientific_limitations': [

        'Physical scale, intrinsics and independent '
        'accuracy not certified',

        'Tests use PothRGBD annotations, not '
        'actual teammate model predictions',

        'Pixel crack topology has sampling '
        'and connectivity limitations',

        'Selected checkpoint was chosen using '
        'development data, not an independent test'
    ]
}

savejson(
    RUN / 'release_candidate_config.json',
    config
)


# ================================================================
# 15. SAVE SOFTWARE TEST EVIDENCE
# ================================================================

savejson(

    RUN / 'software_contract_smoke.json',

    {

        'synthetic_frame_bbox_equivalence':
            'PASS',

        'real_input_tests':
            len(real),

        'cold_loaded_checkpoint':
            'PASS',

        'invalid_input_tests':
            len(invalid_tests),

        'empty_mask_test':
            'PASS',

        'physical_measurement_abstention':
            'PASS',

        'severity_abstention':
            'PASS'
    }
)

savejson(
    RUN / 'real_pothrgbd_training_role_smoke.json',
    real
)


# ================================================================
# 16. SAVE BACKEND API CONTRACT
# ================================================================

savejson(

    RUN / 'api_contract.json',

    {

        'schema_version':
            'crdl-c3/0.8-development',

        'constructor':
            'Component3Handoff(bundle_path, device="cpu")',

        'method':
            'predict_frame(rgb_uint8, defects, '
            'include_maps=False)',

        'defect_fields': [

            'defect_id',

            'bbox_xyxy',

            'mask_space',

            'mask',

            'detection_confidence (optional)',

            'segmentation_confidence (optional)',

            'crack_mask (optional, full frame)',

            'crack_confidence_map (optional, full frame)'
        ],

        'bbox_coords':
            'xyxy pixel, half-open',

        'mask_space':

            'frame: mask HxW; '
            'bbox: ROI mask (y1-y0)x(x1-x0)',

        'outputs': [

            'geometry_raw',

            'pixel_measurements',

            'physical_measurements: null',

            'severity: unassigned',

            'reliability_status',

            'reliability',

            'reasons'
        ],

        'C1_C2_integration_level':
            'Schema adapter and synthetic/real '
            'annotation smoke tests only'
    }
)


# ================================================================
# 17. GENERATE ARTIFACT CHECKSUMS
# ================================================================

receipts = []

for p in sorted(
    RUN.iterdir()
):

    if (
        p.is_file()
        and p.name != 'sha256_manifest.json'
    ):

        receipts.append({

            'file': p.name,

            'bytes': p.stat().st_size,

            'sha256': sha(p)
        })

savejson(
    RUN / 'sha256_manifest.json',
    receipts
)

assert all(

    sha(SOURCE / n) == original_hashes[n]

    for n in FILES
)


# ================================================================
# 18. FINAL SECTION SUMMARY
# ================================================================

print("\n" + "=" * 70)

print(
    "C3-18 SECTION 1 — "
    "RELIABILITY AND C1/C2 HANDOFF PROTOTYPE COMPLETE"
)

print("=" * 70)

print(
    "Frozen trained checkpoint load: PASS"
)

print(
    "Synthetic frame/ROI handoff: PASS"
)

print(
    "Pixel crack geometry and confidence inputs: PASS"
)

print(
    "Input validation and empty-mask safeguards: PASS"
)

print(
    "Real PothRGBD training examples:",
    len(real),
    "PASS"
)

print(
    "Metric geometry measurements certified: NO"
)

print(
    "Physical and visual severity rules validated: NO"
)

print(
    "Actual teammate C1/C2 inference handoff tested: NO"
)

print(
    "Source bundle hashes unchanged: YES"
)

print("\nSaved:")
print(RUN)

print("=" * 70)

Actual C3-14 trained checkpoint: COLD-LOADED
Frame/ROI equivalence, invalid inputs, empty mask, crack inputs and abstention: PASS

C3-18 SECTION 1 — RELIABILITY AND C1/C2 HANDOFF PROTOTYPE COMPLETE
Frozen trained checkpoint load: PASS
Synthetic frame/ROI handoff: PASS
Pixel crack geometry and confidence inputs: PASS
Input validation and empty-mask safeguards: PASS
Real PothRGBD training examples: 3 PASS
Metric geometry measurements certified: NO
Physical and visual severity rules validated: NO
Actual teammate C1/C2 inference handoff tested: NO
Source bundle hashes unchanged: YES

Saved:
/content/drive/MyDrive/crdl_experiments/component3_fresh_v1/08_integration/C3-18_reliability_C1C2_handoff_20261021T090221Z_3946b7


In [3]:
# ================================================================
# CRDL C3-19 — RELEASE CANDIDATE ACCEPTANCE + AUDIT
# ONE COLAB CELL — NO UPLOADS OR RETRAINING
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import csv, json, hashlib, shutil, uuid
import importlib.util, zipfile, platform

import numpy as np
import torch
import scipy
import skimage
import PIL
from PIL import Image, ImageDraw


# ================================================================
# 1. RESTORE EXACT SAVED PROJECT LOCATIONS
# ================================================================

ROOT = Path('/content/drive/MyDrive')

if not ROOT.is_dir():
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = (
    ROOT / 'crdl_experiments' / 'component3_fresh_v1'
)

DATA = (
    ROOT / 'manushi_research_data'
    / 'pothrgbd' / 'PData-main'
)

SOURCE = (
    PROJECT / '08_integration'
    / 'C3-18_reliability_C1C2_handoff_20261021T090221Z_3946b7'
)

C310 = (
    PROJECT / '07_evaluation'
    / 'C3-09_physical_evidence_gate_v2_20261021T024723Z_ec15d662'
    / 'C3-10_section02_grouped_pilot_protocol_20261021T042524Z_735c9f'
)

ROLE_FILE = (
    C310 / 'PROVISIONAL_pilot_role_manifest.csv'
)

LABEL_FILE = (
    PROJECT / '04_geometry_targets'
    / 'C3-11_section01_RAW_target_pilot_20261021T043311Z_367db5'
    / 'C3-11_section04_weighted_feasibility_20261021T050356Z_383980'
    / 'target_eligibility_manifest.csv'
)

C313 = (
    PROJECT / '07_evaluation'
    / 'C3-13_section01_frozen_dev_20261021T063918Z_9fee2e'
    / 'frozen_dev_summary.json'
)

C315 = (
    PROJECT / '05_training_runs'
    / 'C3-15_depth_first_baseline_20261021T071213Z_af6506'
    / 'depth_first_comparison.json'
)

REQUIRED = [
    'selected_model.pt',
    'model_and_inference.py',
    'component3_service.py',
    'geometry_measurements.py',
    'component3_measurement_service.py',
    'component3_handoff_service.py',
    'bundle_config.json',
    'release_candidate_config.json',
    'software_contract_smoke.json'
]

for p in [
    ROLE_FILE, LABEL_FILE, C313, C315,
    SOURCE / 'sha256_manifest.json'
] + [SOURCE / n for n in REQUIRED]:

    assert p.is_file(), f'Missing prerequisite: {p}'


# ================================================================
# 2. CHECKSUM AND REPORT HELPERS
# ================================================================

def sha(path):

    h = hashlib.sha256()

    with Path(path).open('rb') as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b''
        ):
            h.update(block)

    return h.hexdigest()


def readcsv(path):

    with Path(path).open(
        newline='',
        encoding='utf-8'
    ) as f:
        return list(csv.DictReader(f))


def write_json(path, obj):

    Path(path).write_text(
        json.dumps(
            obj, indent=2, allow_nan=False
        ),
        encoding='utf-8'
    )


# ================================================================
# 3. VERIFY PREVIOUS BUNDLE INTEGRITY
# ================================================================

source_manifest = json.loads(
    (SOURCE / 'sha256_manifest.json').read_text()
)

assert len(source_manifest) > 0

for entry in source_manifest:

    p = SOURCE / entry['file']

    assert p.is_file()
    assert sha(p) == entry['sha256'], (
        f'C3-18 integrity failed: {p.name}'
    )

source_hashes = {
    n: sha(SOURCE / n)
    for n in REQUIRED
}

source_hashes['sha256_manifest.json'] = sha(
    SOURCE / 'sha256_manifest.json'
)


# ================================================================
# 4. CREATE A NEW VERSIONED RELEASE FOLDER
# ================================================================

RUN = (
    PROJECT / '08_integration'
    / (
        'C3-19_release_candidate_'
        + datetime.now(
            timezone.utc
        ).strftime('%Y%m%dT%H%M%SZ_')
        + uuid.uuid4().hex[:6]
    )
)

RUN.mkdir(
    parents=True,
    exist_ok=False
)

for name in REQUIRED:
    shutil.copy2(SOURCE / name, RUN / name)

assert all(
    sha(RUN / n) == source_hashes[n]
    for n in REQUIRED
)

print('=' * 69)
print('CRDL C3-19 — RELEASE CANDIDATE ACCEPTANCE')
print('=' * 69)
print('Source integrity: PASS')
print('Output:', RUN)


# ================================================================
# 5. CREATE A REUSABLE C1/C2 BACKEND BRIDGE
# ================================================================

BRIDGE_SOURCE = r'''
from pathlib import Path
import importlib.util


class Component3Bridge:

    """Development-stage C1/C2-to-C3 schema adapter."""

    def __init__(self, root, device='cpu'):

        root = Path(root)

        spec = importlib.util.spec_from_file_location(
            'c319_saved_handoff',
            root / 'component3_handoff_service.py'
        )

        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)

        self.service = mod.Component3Handoff(
            root,
            device=device
        )

    def predict(
        self,
        rgb_uint8,
        detections,
        include_maps=False
    ):

        if not isinstance(detections, (list, tuple)):
            raise ValueError('detections must be a list')

        adapted = []

        for d in detections:

            if not isinstance(d, dict):
                raise ValueError('Invalid detection record')

            required = (
                'defect_id',
                'bbox_xyxy',
                'mask_space',
                'mask'
            )

            missing = [
                k for k in required
                if k not in d
            ]

            if missing:
                raise ValueError(
                    'Missing required keys: '
                    + ','.join(missing)
                )

            if any(
                k in d
                for k in (
                    'mask_probabilities',
                    'mask_logits',
                    'bbox_xywh',
                    'bbox_xywh_norm'
                )
            ):
                raise ValueError(
                    'Convert probabilities and bbox formats '
                    'explicitly before Component 3'
                )

            if (
                'score' in d
                and 'detection_confidence' in d
            ):
                raise ValueError(
                    'Ambiguous detection confidence'
                )

            if (
                'segmentation_score' in d
                and 'segmentation_confidence' in d
            ):
                raise ValueError(
                    'Ambiguous segmentation confidence'
                )

            p = {
                k: d[k]
                for k in required
            }

            p['detection_confidence'] = d.get(
                'detection_confidence',
                d.get('score')
            )

            p['segmentation_confidence'] = d.get(
                'segmentation_confidence',
                d.get('segmentation_score')
            )

            for k in (
                'crack_mask',
                'crack_confidence_map'
            ):
                if k in d:
                    p[k] = d[k]

            adapted.append(p)

        return self.service.predict_frame(
            rgb_uint8,
            adapted,
            include_maps=include_maps
        )
'''

(RUN / 'component3_bridge.py').write_text(
    BRIDGE_SOURCE,
    encoding='utf-8'
)


# ================================================================
# 6. COLD-LOAD ACTUAL TRAINED MODEL THROUGH BRIDGE
# ================================================================

spec = importlib.util.spec_from_file_location(
    'c319_bridge',
    RUN / 'component3_bridge.py'
)

mod = importlib.util.module_from_spec(spec)

spec.loader.exec_module(mod)

DEVICE = (
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

bridge = mod.Component3Bridge(
    RUN,
    device=DEVICE
)

print('Frozen trained checkpoint cold-load: PASS')
print('Device:', DEVICE)


# ================================================================
# 7. ACCEPTANCE TEST HELPERS
# ================================================================

tests = []

def check(name, fn):
    fn()
    tests.append({
        'test': name,
        'status': 'PASS'
    })


def must_raise(fn):

    try:
        fn()
    except ValueError:
        return

    raise AssertionError('Invalid input was accepted')


def assert_true(condition):
    assert condition


def assert_abstention(out):

    assert out['severity']['physical_level'] is None
    assert out['severity']['visual_level'] is None

    assert all(
        v is None
        for v in out['physical_measurements'].values()
    )

    assert out['reliability_status'] in (
        'REVIEW_ONLY_UNCALIBRATED',
        'REVIEW_REQUIRED_INPUT_QA',
        'UNSUPPORTED_EMPTY_MASK'
    )


# ================================================================
# 8. PREPARE SYNTHETIC ACCEPTANCE INPUTS
# ================================================================

h, w = 192, 256

y, x = np.indices((h, w))

rgb = np.zeros(
    (h, w, 3),
    dtype=np.uint8
)

rgb[..., 0] = x.astype(np.uint8)
rgb[..., 1] = y.astype(np.uint8)
rgb[..., 2] = 75

mask = (
    ((x - 105) / 36) ** 2
    + ((y - 88) / 27) ** 2
) < 1

ys, xs = np.where(mask)

box = [
    int(xs.min()),
    int(ys.min()),
    int(xs.max() + 1),
    int(ys.max() + 1)
]

roi = mask[
    box[1]:box[3],
    box[0]:box[2]
]

crack = np.zeros(
    (h, w),
    dtype=np.uint8
)

crack[75:101, 104:107] = 1

p1 = {
    'defect_id': 'synthetic-a',
    'bbox_xyxy': box,
    'mask_space': 'frame',
    'mask': mask,
    'score': 0.8,
    'segmentation_score': 0.9,
    'crack_mask': crack,
    'crack_confidence_map': np.full(
        (h, w), 0.7
    )
}

p2 = {
    **p1,
    'mask_space': 'bbox',
    'mask': roi
}


# ================================================================
# 9. FULL-FRAME VERSUS ROI ACCEPTANCE
# ================================================================

a = bridge.predict(rgb, [p1])['defects'][0]
b = bridge.predict(rgb, [p2])['defects'][0]

check(
    'frame/ROI equivalence',
    lambda: (
        assert_true(
            a['pixel_measurements']['footprint_area_px2']
            ==
            b['pixel_measurements']['footprint_area_px2']
        ),
        assert_true(
            np.isclose(
                a['geometry_raw']['d95_raw'],
                b['geometry_raw']['d95_raw'],
                atol=1e-6
            )
        )
    )
)


# ================================================================
# 10. CRACK AND SEVERITY SAFETY TEST
# ================================================================

check(
    'crack metrology and safe severity',
    lambda: (
        assert_true(
            a['pixel_measurements']['crack']
            ['skeleton_edge_length_px_approx'] > 0
        ),
        assert_abstention(a)
    )
)


# ================================================================
# 11. INVALID INPUT TESTS
# ================================================================

check(
    'reject probability mask',
    lambda: must_raise(
        lambda: bridge.predict(
            rgb,
            [{
                **p1,
                'mask': np.full((h, w), 0.4)
            }]
        )
    )
)

check(
    'reject out-of-range confidence',
    lambda: must_raise(
        lambda: bridge.predict(
            rgb,
            [{**p1, 'score': 1.5}]
        )
    )
)

check(
    'reject mismatched ROI',
    lambda: must_raise(
        lambda: bridge.predict(
            rgb,
            [{
                **p1,
                'mask_space': 'bbox',
                'mask': np.zeros((4, 5))
            }]
        )
    )
)

check(
    'reject ambiguous bbox format',
    lambda: must_raise(
        lambda: bridge.predict(
            rgb,
            [{
                **p1,
                'bbox_xywh_norm': [0.2, 0.2, 0.4, 0.4]
            }]
        )
    )
)

check(
    'reject duplicate IDs',
    lambda: must_raise(
        lambda: bridge.predict(
            rgb,
            [p1, p1]
        )
    )
)

check(
    'reject non-uint8 RGB',
    lambda: must_raise(
        lambda: bridge.predict(
            rgb.astype(np.float32),
            [p1]
        )
    )
)


# ================================================================
# 12. EMPTY-MASK SAFETY TEST
# ================================================================

empty = {
    **p1,
    'defect_id': 'synthetic-empty',
    'mask': np.zeros((h, w), dtype=bool)
}

empty_result = bridge.predict(
    rgb, [empty]
)['defects'][0]

check(
    'empty-mask abstention',
    lambda: (
        assert_true(
            empty_result['geometry_raw'] is None
        ),
        assert_true(
            empty_result['reliability_status']
            == 'UNSUPPORTED_EMPTY_MASK'
        ),
        assert_abstention(empty_result)
    )
)


# ================================================================
# 13. MULTIPLE DEFECTS IN ONE FRAME
# ================================================================

mask2 = (
    ((x - 190) / 16) ** 2
    + ((y - 130) / 13) ** 2
) < 1

y2, x2 = np.where(mask2)

box2 = [
    int(x2.min()),
    int(y2.min()),
    int(x2.max() + 1),
    int(y2.max() + 1)
]

psecond = {
    'defect_id': 'synthetic-b',
    'bbox_xyxy': box2,
    'mask_space': 'frame',
    'mask': mask2,
    'score': 0.7,
    'segmentation_score': 0.75
}

multi = bridge.predict(
    rgb, [p1, psecond]
)

check(
    'two-defect frame',
    lambda: (
        assert_true(multi['defect_count'] == 2),
        assert_true(
            [d['defect_id'] for d in multi['defects']]
            == ['synthetic-a', 'synthetic-b']
        )
    )
)


# ================================================================
# 14. VERIFY OPTIONAL DENSE GEOMETRY OUTPUT
# ================================================================

fullmap = bridge.predict(
    rgb,
    [p1],
    include_maps=True
)['defects'][0]

arr = np.asarray(
    fullmap['raw_axial_depression_map']
)

smallmask = np.asarray(
    Image.fromarray(
        mask.astype('uint8')
    ).resize(
        (256, 192),
        Image.Resampling.NEAREST
    ),
    dtype=bool
)

check(
    'dense map validity',
    lambda: (
        assert_true(arr.shape == (192, 256)),
        assert_true(np.isfinite(arr).all()),
        assert_true((arr >= 0).all()),
        assert_true((arr[~smallmask] == 0).all())
    )
)


# ================================================================
# 15. YOLO POLYGON DECODER
# ================================================================

def decode_yolo(path, h, w):

    canvas = Image.new(
        'L', (w, h), 0
    )

    draw = ImageDraw.Draw(canvas)

    count = 0

    for line in Path(path).read_text(
        encoding='utf-8'
    ).splitlines():

        if not line.strip():
            continue

        vals = np.array(
            [float(t) for t in line.split()],
            dtype=np.float64
        )

        if not (
            len(vals) >= 7
            and (len(vals) - 1) % 2 == 0
            and np.isfinite(vals).all()
            and vals[0] == 0
            and np.all(
                (vals[1:] >= 0)
                & (vals[1:] <= 1)
            )
        ):
            raise ValueError(
                'Invalid class-0 polygon'
            )

        points = vals[1:].reshape(-1, 2)

        draw.polygon(
            [
                (
                    round(float(px) * (w - 1)),
                    round(float(py) * (h - 1))
                )
                for px, py in points
            ],
            fill=1
        )

        count += 1

    if count < 1:
        raise ValueError('Missing polygon')

    return np.asarray(canvas, dtype=bool)


# ================================================================
# 16. RESTORE STRICT TRAINING-ROLE EXAMPLES
# ================================================================

roles = {
    r['sample_id']: r
    for r in readcsv(ROLE_FILE)
}

labels = readcsv(LABEL_FILE)

strict_fit = sorted(
    [
        r for r in labels
        if (
            r['tier'] == 'STRICT'
            and roles[
                r['sample_id']
            ]['temporal_burst_proxy']
            == 'POTHRGBD_TEMPORAL_0004'
        )
    ],
    key=lambda r: r['sample_id']
)

assert len(strict_fit) == 79


# ================================================================
# 17. LOAD ORIGINAL C3-18 FOR OUTPUT EQUIVALENCE
# ================================================================

original_spec = importlib.util.spec_from_file_location(
    'c319_original',
    SOURCE / 'component3_handoff_service.py'
)

original_module = importlib.util.module_from_spec(
    original_spec
)

original_spec.loader.exec_module(
    original_module
)

original = original_module.Component3Handoff(
    SOURCE,
    device=DEVICE
)


# ================================================================
# 18. TEST TWELVE REAL POTHRGBD INPUTS
# ================================================================

real = []

for i, row in enumerate(strict_fit[:12], 1):

    sid = row['sample_id']
    entry = roles[sid]

    imagepath = Path(entry['rgb_path'])
    labelpath = Path(entry['label_path'])

    assert imagepath.is_file()
    assert labelpath.is_file()

    assert imagepath.parent.resolve() == (
        DATA / 'images'
    ).resolve()

    assert labelpath.parent.resolve() == (
        DATA / 'labels'
    ).resolve()

    with Image.open(imagepath) as im:
        real_rgb = np.asarray(
            im.convert('RGB'),
            dtype=np.uint8
        )

    real_mask = decode_yolo(
        labelpath,
        *real_rgb.shape[:2]
    )

    assert real_mask.any()

    yy, xx = np.where(real_mask)

    bb = [
        int(xx.min()),
        int(yy.min()),
        int(xx.max() + 1),
        int(yy.max() + 1)
    ]

    mode = (
        'bbox'
        if i % 2 == 0
        else 'frame'
    )

    supplied_mask = (
        real_mask[
            bb[1]:bb[3],
            bb[0]:bb[2]
        ]
        if mode == 'bbox'
        else real_mask
    )

    packet = {
        'defect_id': sid,
        'bbox_xyxy': bb,
        'mask_space': mode,
        'mask': supplied_mask
    }

    new = bridge.predict(
        real_rgb,
        [packet]
    )['defects'][0]

    reference = original.predict_frame(
        real_rgb,
        [{
            'defect_id': sid,
            'bbox_xyxy': bb,
            'mask_space': 'frame',
            'mask': real_mask
        }]
    )['defects'][0]

    assert np.isclose(
        new['geometry_raw']['d95_raw'],
        reference['geometry_raw']['d95_raw'],
        atol=1e-5,
        rtol=1e-6
    )

    assert (
        new['pixel_measurements']
        ['footprint_area_px2']
        == int(real_mask.sum())
    )

    assert_abstention(new)

    real.append({
        'sample_id': sid,
        'mask_space': mode,
        'mask_pixels': int(real_mask.sum()),
        'd95_raw': new['geometry_raw']['d95_raw'],
        'status': new['reliability_status'],
        'matches_original_bundle': True
    })

    if i % 4 == 0:
        print(
            f'Real PothRGBD acceptance: {i}/12',
            flush=True
        )

check(
    '12 annotated real inputs vs frozen C3-18',
    lambda: assert_true(len(real) == 12)
)


# ================================================================
# 19. VERIFY PREVIOUS DEVELOPMENT EVIDENCE
# ================================================================

eval_summary = json.loads(
    C313.read_text(encoding='utf-8')
)

depth_summary = json.loads(
    C315.read_text(encoding='utf-8')
)

assert eval_summary['source_count'] == 275
assert eval_summary['n_scored_strict'] == 72

assert depth_summary['source_references'] == 72
assert depth_summary['scored'] == 72

assert eval_summary['metric_units_verified'] is False


# ================================================================
# 20. SAVE SOFTWARE ACCEPTANCE RESULTS
# ================================================================

write_json(
    RUN / 'acceptance_tests.json',
    {
        'total_passed': len(tests),
        'tests': tests,
        'real_annotated_training_examples': real,
        'teammate_prediction_inputs_tested': False
    }
)


# ================================================================
# 21. SAVE BACKEND API CONTRACT
# ================================================================

write_json(
    RUN / 'api_contract.json',
    {
        'version': 'crdl-c3/0.9-release-candidate',

        'call': (
            'Component3Bridge(bundle_directory, device)'
            '.predict(rgb_uint8, detections, '
            'include_maps=False)'
        ),

        'bbox': 'pixel xyxy half-open integers',

        'mask_space': [
            'frame: HxW',
            'bbox: (y1-y0)x(x1-x0)'
        ],

        'detector_fields': [
            'defect_id',
            'bbox_xyxy',
            'mask_space',
            'mask',
            'score / detection_confidence (optional)',
            'segmentation_score / segmentation_confidence (optional)',
            'crack_mask (optional full-frame)',
            'crack_confidence_map (optional full-frame)'
        ],

        'probability_maps': (
            'Explicitly threshold in C2; '
            'no implicit conversion'
        ),

        'output': (
            'Per-defect geometry_raw, '
            'pixel_measurements, QA/reliability, '
            'null physical measurements and severity'
        ),

        'units': 'UNVERIFIED_RAW_AXIAL and pixel',

        'actual_C1C2_handoff': 'NOT YET VERIFIED'
    }
)


# ================================================================
# 22. SAVE QUANTITATIVE RESEARCH EVIDENCE
# ================================================================

write_json(
    RUN / 'research_evidence_summary.json',
    {
        'primary_model': (
            'DIRECT_RGB_MASK frozen C3-14/C3-18 checkpoint'
        ),

        'development_total': 275,

        'development_strict_scored': 72,

        'development_strict_coverage': 72 / 275,

        'model_MAE_RAW':
            eval_summary['score_summary']
            ['DIRECT_RGB_MASK']['mae_raw'],

        'mask_distance_MAE_RAW':
            eval_summary['score_summary']
            ['TRAIN_MASK_DISTANCE']['mae_raw'],

        'depth_first_MAE_RAW':
            depth_summary['depth_first_mae_raw'],

        'depth_first_comparisons':
            depth_summary['scored'],

        'independent_field_validation': False,

        'metric_depth_verified': False,

        'real_severity_validated': False,

        'synthetic_xyz_evidence': (
            'C3-17 passed; not a real-camera '
            'calibration validation'
        ),

        'known_weaknesses': [
            'Only 72 quality-selected development references',
            'Model selected using development data',
            'Physical camera scale and intrinsics unverified',
            'Depth-first baseline exhibited weak predictions',
            'Actual teammate model outputs not tested',
            'Pixel crack geometry is approximate',
            'No validated severity thresholds'
        ]
    }
)


# ================================================================
# 23. ASSESSMENT RUBRIC EVIDENCE MATRIX
# ================================================================

write_json(
    RUN / 'rubric_evidence_matrix.json',
    {
        'problem_definition':
            'Road-relative depression geometry and '
            'measurement-eligibility problem documented',

        'proof_of_concept':
            'Trained checkpoint, 72 strict development '
            'evaluations, and end-to-end adapter tests',

        'research_pillars':
            'Supervised computer vision, inverse-depth '
            'reference modelling, geometry and QA',

        'technology_application':
            'PyTorch, learned depth, segmentation masks, '
            'pixel metrology and conditional 3D',

        'design_excellence':
            'Versioned release, API schema, checksums, '
            'safety gates and modular architecture',

        'prototype_completion':
            'Development prototype completed; '
            'physical validation outstanding',

        'standards_practices':
            'Frozen inference, hash integrity, '
            'leakage-aware data grouping, honest units',

        'user_requirements':
            'Per-defect measurements, optional cracks, '
            'review status and explicit unsupported fields',

        'risk_mitigation':
            'Abstention on unsupported physical outputs, '
            'invalid inputs rejected',

        'remaining': [
            'Actual teammate C1/C2 outputs',
            'Capture-specific physical calibration',
            'Independent physical measurement evaluation',
            'Validated severity policy'
        ]
    }
)


# ================================================================
# 24. ENVIRONMENT AND DEPENDENCY RECORD
# ================================================================

write_json(
    RUN / 'environment.json',
    {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'numpy': np.__version__,
        'scipy': scipy.__version__,
        'scikit_image': skimage.__version__,
        'pillow': PIL.__version__,
        'device_used': DEVICE,

        'source_C318': str(SOURCE),

        'selected_model_sha256':
            source_hashes['selected_model.pt'],

        'source_manifest_sha256':
            source_hashes['sha256_manifest.json'],

        'role_manifest_sha256': sha(ROLE_FILE),

        'label_manifest_sha256': sha(LABEL_FILE),

        'evaluation_summary_sha256': sha(C313),

        'depth_first_summary_sha256': sha(C315)
    }
)

(RUN / 'requirements_colab.txt').write_text(
    '\n'.join([
        f'numpy=={np.__version__}',
        f'scipy=={scipy.__version__}',
        f'scikit-image=={skimage.__version__}',
        f'Pillow=={PIL.__version__}',
        f'torch=={torch.__version__}'
    ]) + '\n',
    encoding='utf-8'
)


# ================================================================
# 25. RELEASE USAGE DOCUMENTATION
# ================================================================

README = """
# CRDL C3-19 — Development Release Candidate

Extract component3_release_candidate.zip into one folder,
or use this experiment folder directly.

Example Python integration:

    from component3_bridge import Component3Bridge

    bridge = Component3Bridge(
        "/path/to/extracted/folder",
        device="cpu"
    )

    result = bridge.predict(
        rgb_uint8,
        detections=[
            {
                "defect_id": "pothole-1",
                "bbox_xyxy": [x0, y0, x1, y1],
                "mask_space": "frame",
                "mask": binary_HxW_mask
            }
        ]
    )

Input conventions:
- RGB must be HxWx3 uint8.
- Bounding boxes are half-open xyxy pixel coordinates.
- Segmentation masks must be aligned and binary.
- For a cropped mask, use mask_space="bbox".
- Optional score and segmentation_score are uncalibrated.
- Crack masks must be full-frame aligned.

IMPORTANT:
This is a development prototype, not a physically certified
road safety measurement system.

Raw geometry is NOT millimetres.
Physical measurements and severity remain unavailable.
Independent field validation and actual teammate prediction
handoff remain outstanding.

The original trained CRDL checkpoint is preserved.
"""

(RUN / 'README.md').write_text(
    README,
    encoding='utf-8'
)


# ================================================================
# 26. PACKAGE RELEASE INTO ONE ZIP INSIDE DRIVE
# ================================================================

package_names = REQUIRED + [
    'component3_bridge.py',
    'README.md',
    'api_contract.json',
    'research_evidence_summary.json',
    'rubric_evidence_matrix.json',
    'environment.json',
    'acceptance_tests.json',
    'requirements_colab.txt'
]

ZIP_FILE = (
    RUN / 'component3_release_candidate.zip'
)

with zipfile.ZipFile(
    ZIP_FILE,
    'w',
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as archive:

    for name in package_names:
        archive.write(
            RUN / name,
            arcname=name
        )

with zipfile.ZipFile(ZIP_FILE) as archive:

    assert archive.testzip() is None

    assert set(archive.namelist()) == set(
        package_names
    )


# ================================================================
# 27. GENERATE FINAL SHA256 MANIFEST
# ================================================================

receipts = []

for p in sorted(RUN.iterdir()):

    if (
        p.is_file()
        and p.name != 'sha256_manifest.json'
    ):

        receipts.append({
            'file': p.name,
            'bytes': p.stat().st_size,
            'sha256': sha(p)
        })

write_json(
    RUN / 'sha256_manifest.json',
    receipts
)

assert all(
    sha(SOURCE / n) == source_hashes[n]
    for n in REQUIRED
)


# ================================================================
# 28. FINAL C3-19 ACCEPTANCE SUMMARY
# ================================================================

print('\n' + '=' * 69)
print(
    'C3-19 — DEVELOPMENT RELEASE CANDIDATE '
    'ACCEPTANCE COMPLETE'
)
print('=' * 69)

print('Cold-loaded original trained model: PASS')

print('Passed acceptance checks:', len(tests))

print(
    'Real PothRGBD annotated training inputs:',
    len(real),
    'PASS'
)

print('Frozen C3-18 output equivalence: PASS')

print(
    'Development group: 275 total; '
    '72 strict scored (26.2%)'
)

print('Physical scale and severity certification: NO')

print('Real teammate predicted outputs integrated: NO')

print('Independent physical field validation: NO')

print('Original C3-18 source hashes unchanged: YES')

print('Release ZIP, API, QA and evidence: SAVED')

print('\nSaved:')
print(RUN)

print('=' * 69)

CRDL C3-19 — RELEASE CANDIDATE ACCEPTANCE
Source integrity: PASS
Output: /content/drive/MyDrive/crdl_experiments/component3_fresh_v1/08_integration/C3-19_release_candidate_20261021T091823Z_651136
Frozen trained checkpoint cold-load: PASS
Device: cpu
Real PothRGBD acceptance: 4/12
Real PothRGBD acceptance: 8/12
Real PothRGBD acceptance: 12/12

C3-19 — DEVELOPMENT RELEASE CANDIDATE ACCEPTANCE COMPLETE
Cold-loaded original trained model: PASS
Passed acceptance checks: 12
Real PothRGBD annotated training inputs: 12 PASS
Frozen C3-18 output equivalence: PASS
Development group: 275 total; 72 strict scored (26.2%)
Physical scale and severity certification: NO
Real teammate predicted outputs integrated: NO
Independent physical field validation: NO
Original C3-18 source hashes unchanged: YES
Release ZIP, API, QA and evidence: SAVED

Saved:
/content/drive/MyDrive/crdl_experiments/component3_fresh_v1/08_integration/C3-19_release_candidate_20261021T091823Z_651136
